# Train SigExt — Full 27-Model Matrix

Trains **27 SigExt models** across the complete experimental matrix:

| Language | Base Model | Sample Sizes | Thresholds |
|---|---|---|---|
| **English** | `allenai/longformer-base-4096` | 1k, 2.5k, 5k | 0.60, 0.70, 0.80 |
| **English** | `markussagen/xlm-roberta-longformer-base-4096` | 1k, 2.5k, 5k | 0.60, 0.70, 0.80 |
| **Italian** | `markussagen/xlm-roberta-longformer-base-4096` | 1k, 2.5k, 5k | 0.60, 0.70, 0.80 |

**Seed**: 42 (fixed for all). All models pushed to HuggingFace Hub.

**Checkpointing**: Progress is saved to `training_checkpoint.json`. If the notebook is interrupted, rerunning it will skip already completed models.

In [1]:
import warnings, os, json, logging
from dotenv import load_dotenv

# Standard warning suppression
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Silence libraries logging
try:
    import transformers, datasets
    transformers.utils.logging.set_verbosity_error()
    transformers.utils.logging.disable_progress_bar()
    datasets.utils.logging.set_verbosity_error()
    datasets.utils.logging.disable_progress_bar()
    logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
except ImportError: pass

!uv pip install -e ../..
load_dotenv()
from huggingface_hub import login
login(token=os.getenv("HF_TOKEN"), add_to_git_credential=False)

Using Python 3.12.12 environment at: /home/marcantoniolopez/Documenti/github/projects/sm-sip/.venv
Resolved 134 packages in 401ms                                       
   Building sm-sip @ file:///home/marcantoniolopez/Documenti/github/projects/sm-
      Built sm-sip @ file:///home/marcantoniolopez/Documenti/github/projects/sm-
Prepared 1 package in 148ms                                              
Uninstalled 1 package in 0.23ms
Installed 1 package in 0.85msle:///home/marcantoniolopez/Doc
 ~ sm-sip==1.0.0 (from file:///home/marcantoniolopez/Documenti/github/projects/sm-sip)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
from sm_sip.config import TrainingConfig, SigExtConfig
from sm_sip.pipelines.training import train_sigext
from sm_sip.utils.gpu import clear_gpu_memory

## Training Matrix (27 configs)

In [3]:
CHECKPOINT_FILE = "training_checkpoint.json"

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            return json.load(f)
    return []

def save_checkpoint(completed_models):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(completed_models, f, indent=4)

completed_models = load_checkpoint()
print(f"Found {len(completed_models)} completed models in checkpoint.")

Found 0 completed models in checkpoint.


In [4]:
configs = []

for lang, ld in SigExtConfig.LANG_DATASETS.items():
    for base_key in SigExtConfig.LANG_BASE_MODELS[lang]:
        base_model_id = SigExtConfig.BASE_MODELS[base_key]
        for n in SigExtConfig.SAMPLE_SIZES:
            for t in SigExtConfig.THRESHOLDS:
                name_n = f'{n // 1000}k' if n >= 1000 and n % 1000 == 0 else str(n)
                t_str = f'{t:.2f}'.replace('.', '')
                output_name = f'sigext-{ld["prefix"]}-{lang}-{base_key}-{name_n}-{t_str}t'
                
                configs.append(TrainingConfig(
                    lang=lang,
                    base_model_id=base_model_id,
                    dataset_name=ld['dataset'],
                    num_samples=n,
                    similarity_threshold=t,
                    output_model_name=output_name,
                    push_to_hub=True,
                    seed=SigExtConfig.SEED,
                ))

print(f'Total training configs: {len(configs)}')
print(f'\n{"Model Name":<50} {"Base":>12} {"Lang":>4} {"N":>6} {"Thr":>5}')
print('-' * 80)
for c in configs:
    base_short = c.base_model_id.split('/')[-1][:12]
    status = "[DONE]" if c.output_model_name in completed_models else ""
    print(f'{c.output_model_name:<50} {base_short:>12} {c.lang:>4} {c.num_samples:>6} {c.similarity_threshold:>5.2f} {status}')

Total training configs: 27

Model Name                                                 Base Lang      N   Thr
--------------------------------------------------------------------------------
sigext-wits-it-xlmr-1k-060t                        xlm-roberta-   it   1000  0.60 
sigext-wits-it-xlmr-1k-070t                        xlm-roberta-   it   1000  0.70 
sigext-wits-it-xlmr-1k-080t                        xlm-roberta-   it   1000  0.80 
sigext-wits-it-xlmr-2500-060t                      xlm-roberta-   it   2500  0.60 
sigext-wits-it-xlmr-2500-070t                      xlm-roberta-   it   2500  0.70 
sigext-wits-it-xlmr-2500-080t                      xlm-roberta-   it   2500  0.80 
sigext-wits-it-xlmr-5k-060t                        xlm-roberta-   it   5000  0.60 
sigext-wits-it-xlmr-5k-070t                        xlm-roberta-   it   5000  0.70 
sigext-wits-it-xlmr-5k-080t                        xlm-roberta-   it   5000  0.80 
sigext-arxiv-en-allenai-1k-060t                    longformer-

## Training Loop

In [ ]:
for i, config in enumerate(configs):
    if config.output_model_name in completed_models:
        print(f"Skip {config.output_model_name} (already completed)")
        continue

    print(f'\n{"="*70}')
    print(f'  [{i+1}/{len(configs)}] {config.output_model_name}')
    print(f'  Base: {config.base_model_id}')
    print(f'  Lang: {config.lang}, Samples: {config.num_samples}, Threshold: {config.similarity_threshold}')
    print(f'  Seed: {config.seed}')
    print(f'{"="*70}')

    try:
        model, tokenizer = train_sigext(config)
        del model, tokenizer
        clear_gpu_memory()
        
        # Update checkpoint
        completed_models.append(config.output_model_name)
        save_checkpoint(completed_models)
        
        print(f'  {config.output_model_name} complete!')
    except Exception as e:
        print(f'  FAILED: {e}')
        clear_gpu_memory()
        continue

print(f'\n\nAll {len(configs)} training runs complete!')


  [1/27] sigext-wits-it-xlmr-1k-060t
  Base: markussagen/xlm-roberta-longformer-base-4096
  Lang: it, Samples: 1000, Threshold: 0.6
  Seed: 42
TRAINING: sigext-wits-it-xlmr-1k-060t (seed=42)
  Loading dataset: silvia-casola/WITS...


Repo card metadata block was not found. Setting CardData to empty.


  Preparing training data...


Preparing training data:   2%|▏         | 16/1000 [00:56<58:11,  3.55s/it]


  FAILED: CUDA out of memory. Tried to allocate 26.00 MiB. GPU 0 has a total capacity of 7.53 GiB of which 10.00 MiB is free. Including non-PyTorch memory, this process has 7.50 GiB memory in use. Of the allocated memory 7.08 GiB is allocated by PyTorch, and 240.18 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

  [2/27] sigext-wits-it-xlmr-1k-070t
  Base: markussagen/xlm-roberta-longformer-base-4096
  Lang: it, Samples: 1000, Threshold: 0.7
  Seed: 42
TRAINING: sigext-wits-it-xlmr-1k-070t (seed=42)
  Loading dataset: silvia-casola/WITS...


...

Preparing training data:   3%|▎         | 29/1000 [01:29<50:00,  3.09s/it]


  FAILED: CUDA out of memory. Tried to allocate 174.00 MiB. GPU 0 has a total capacity of 7.53 GiB of which 120.00 MiB is free. Including non-PyTorch memory, this process has 7.39 GiB memory in use. Of the allocated memory 6.69 GiB is allocated by PyTorch, and 536.43 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

  [4/27] sigext-wits-it-xlmr-2500-060t
  Base: markussagen/xlm-roberta-longformer-base-4096
  Lang: it, Samples: 2500, Threshold: 0.6
  Seed: 42
TRAINING: sigext-wits-it-xlmr-2500-060t (seed=42)
  Loading dataset: silvia-casola/WITS...


Repo card metadata block was not found. Setting CardData to empty.


  Preparing training data...


Preparing training data:   0%|          | 1/2500 [00:02<2:00:13,  2.89s/it]